# RAG, FrontierAgent, EnsembleAgent

## Order

1. Modal.com and SpecialistAgent  
2. RAG, FrontierAgent, EnsembleAgent  
3. ScannerAgent, MessagingAgent  
4. AutonomousPlanningAgent and DealAgentFramework  
5. Gradio finale

## RAG over ~88K curated wine tasting notes

The fine-tuned specialist beats a frontier LLM at VFM because it learned our
construct from examples. Now we try an **inference-time** technique instead of
training: give the frontier model the 5 most similar wines (with their real
critic scores and prices) as context, let it estimate points and price for the
target wine, and map those through the same frozen `compute_vfm`.

In [ ]:
import logging
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import chromadb
import joblib
import modal
import numpy as np
import plotly.graph_objects as go
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LinearRegression
from sklearn.manifold import TSNE
from tqdm import tqdm

from utils.deep_neural_network import VfmTrainer
from utils.evaluator import evaluate
from utils.items import Wine
from utils.vfm import MAX_POINTS, MAX_PRICE, MIN_POINTS, MIN_PRICE, compute_vfm

In [ ]:
load_dotenv(override=True)
DB = "wine_vectorstore"
DATASET = "gtraskas/wine-vfm-curated"

The curated dataset is public on the HuggingFace Hub, so no HF login is needed
here. You DO need `OPENAI_API_KEY` in your `.env` for the frontier calls, and
Modal tokens for the specialist.

In [ ]:
train, val, test = Wine.from_hub(DATASET)

print(f"Loaded {len(train):,} train, {len(val):,} validation, {len(test):,} test wines")

# Create a Chroma datastore

Free, open-source vector database — we store all ~88K training-wine summaries
with their points, price, VFM and country as metadata.

In [ ]:
client = chromadb.PersistentClient(path=DB)

# The SentenceTransformer encoder

`all-MiniLM-L6-v2` maps sentences to 384-dimensional vectors and is ideal for
semantic search. It's free, fast, and runs locally. The tasting notes never
leave the machine.

In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
vector = encoder.encode(["A crisp Assyrtiko with citrus, saline minerality and a long finish"])[0]
print(vector.shape)
vector

## Populate the vectorstore

One-time encode of ~88K summaries — expect roughly 10-30 minutes locally
(Apple silicon MPS). The cell is idempotent: it skips if the collection exists.

In [ ]:
collection_name = "wines"
existing_collection_names = [c.name for c in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        batch = train[i : i + 1000]
        documents = [wine.summary or "" for wine in batch]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [
            {
                "points": wine.points,
                "price": wine.price,
                "vfm": wine.vfm,
                "country": wine.country,
            }
            for wine in batch
        ]
        ids = [f"doc_{j}" for j in range(i, i + len(documents))]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

# Visualize the vectorized data

In [ ]:
MAXIMUM_DATAPOINTS = 10_000

In [ ]:
# Color by the 8 most common countries; everything else is gray
result = collection.get(
    include=["embeddings", "documents", "metadatas"], limit=MAXIMUM_DATAPOINTS
)
vectors = np.array(result["embeddings"])
documents = result["documents"]
countries = [metadata["country"] for metadata in result["metadatas"]]

unique, counts = np.unique(countries, return_counts=True)
top_countries = list(unique[np.argsort(counts)[::-1]][:8])
palette = ["cyan", "blue", "brown", "orange", "yellow", "green", "purple", "red"]
color_of = dict(zip(top_countries, palette, strict=False))
colors = [color_of.get(c, "gray") for c in countries]
print(top_countries)

In [ ]:
# t-SNE: t-distributed Stochastic Neighbor Embedding — dimensionality reduction
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(size=4, color=colors, opacity=0.7),
            text=[
                f"Country: {c}<br>Text: {d[:60]}..."
                for c, d in zip(countries, documents, strict=True)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="2D Chroma vectorstore visualization (colored by country)",
    width=1000,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],
            mode="markers",
            marker=dict(size=2, color=colors, opacity=0.7),
            text=[
                f"Country: {c}<br>Text: {d[:60]}..."
                for c, d in zip(countries, documents, strict=True)
            ],
            hoverinfo="text",
        )
    ]
)

fig.update_layout(
    title="3D Chroma vectorstore visualization (colored by country)",
    width=1000,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

# RAG pipeline, step by step

Retrieve the 5 most similar wines, hand their real points and prices to the
LLM as context, get back a points/price estimate for the target wine, and map
it through the frozen `compute_vfm`, the same composite path as the zero-shot
baseline, now grounded by retrieval.

In [ ]:
test[0]

In [ ]:
test[0].summary

In [ ]:
def vector_of(wine: Wine):
    return encoder.encode(wine.summary or "")

In [ ]:
def find_similars(wine: Wine):
    vec = vector_of(wine)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results["documents"][0][:]
    points = [m["points"] for m in results["metadatas"][0][:]]
    prices = [m["price"] for m in results["metadatas"][0][:]]
    return documents, points, prices

In [ ]:
find_similars(test[0])

In [ ]:
def make_context(similars, points, prices):
    message = (
        "For context, here are wines with similar tasting notes, "
        "variety, country, province, and winery, "
        "with their critic scores and prices.\n\n"
    )
    for similar, pts, price in zip(similars, points, prices, strict=True):
        message += (
            f"Possibly related wine:\n{similar}\n"
            f"Critic score: {pts:.0f} points. Price: ${price:.2f}\n\n"
        )
    return message

In [ ]:
documents, points, prices = find_similars(test[0])
print(make_context(documents, points, prices))

In [ ]:
def messages_for(wine: Wine, similars, points, prices):
    instruction = (
        "Estimate this wine's critic score (80-100 points) and its retail "
        "price in USD from the tasting note and details."
    )
    content = f"{instruction}\n\n{wine.summary}\n\n"
    content += make_context(similars, points, prices)
    return [{"role": "user", "content": content}]

In [ ]:
documents, points, prices = find_similars(test[0])
print(messages_for(test[0], documents, points, prices)[0]["content"])

In [ ]:
openai_client = OpenAI()


class PointsPrice(BaseModel):
    points: float
    price: float


def gpt_5_1_rag(wine: Wine) -> int:
    similars, points, prices = find_similars(wine)
    response = openai_client.chat.completions.parse(
        model="gpt-5.1",
        messages=messages_for(wine, similars, points, prices),
        response_format=PointsPrice,
        reasoning_effort="none",
        seed=42,
    )
    estimate = response.choices[0].message.parsed
    if estimate is None:
        return 55  # graceful fallback on malformed output
    pts = min(max(estimate.points, MIN_POINTS), MAX_POINTS)
    price = min(max(estimate.price, MIN_PRICE), MAX_PRICE)
    return compute_vfm(pts, price)

In [ ]:
# The truth for our first test wine
test[0].vfm

In [ ]:
gpt_5_1_rag(test[0])

In [ ]:
evaluate(gpt_5_1_rag, test)

# The specialist, called remotely on Modal

The deployed service from notebook 1 — the fine-tuned Llama 3.2 3B predicting VFM
directly.

In [ ]:
WineSpecialist = modal.Cls.from_name("wine-vfm-specialist-service", "WineSpecialist")
wine_specialist = WineSpecialist()


def specialist(wine: Wine) -> int:
    return wine_specialist.estimate_vfm.remote(wine.summary)

# The neural network

The residual bag-of-words regressor from the baseline ladder. Preferred:
copy the `vfm_net.pth` checkpoint produced by the baseline training notebook
into this folder. The cell below loads it directly.

In [ ]:
trainer = VfmTrainer()

trainer.load("vfm_net.pth")
print("Loaded existing weights")

In [ ]:
def deep_neural_network(wine: Wine) -> float:
    return trainer.predict(wine.summary or "")

# The ensemble: regression-derived weights

Rather than hardcoding blend weights, fit a linear regression on held-out
validation wines: the three sub-model predictions are the features, true VFM
is the target. The fitted model is persisted and loaded by the EnsembleAgent.

In [ ]:
fit_wines = val[:200]


def sub_predictions(wine: Wine) -> list[float]:
    return [specialist(wine), gpt_5_1_rag(wine), deep_neural_network(wine)]


with ThreadPoolExecutor(max_workers=10) as pool:
    rows = list(tqdm(pool.map(sub_predictions, fit_wines), total=len(fit_wines)))

In [ ]:
rows[0]

In [ ]:
fit_wines[0].vfm

In [ ]:
X = np.array(rows)
y = np.array([wine.vfm for wine in fit_wines])

ensemble_model = LinearRegression()
ensemble_model.fit(X, y)

feature_names = ["specialist", "frontier_rag", "neural_network"]
for name, coef in zip(feature_names, ensemble_model.coef_, strict=True):
    print(f"{name:>16}: {coef:+.3f}")
print(f"{'intercept':>16}: {ensemble_model.intercept_:+.3f}")

joblib.dump(ensemble_model, "ensemble_model.joblib")

In [ ]:
def ensemble(wine: Wine) -> float:
    features = np.array([sub_predictions(wine)])
    return float(ensemble_model.predict(features)[0])

In [ ]:
evaluate(ensemble, test)

# And now as proper agents

The same three estimators, wrapped behind the agent interfaces the framework
will orchestrate.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.estimate(test[0].summary)

In [ ]:
from agents.neural_network_agent import NeuralNetworkAgent

agent = NeuralNetworkAgent()
agent.estimate(test[0].summary)

In [ ]:
from agents.ensemble_agent import EnsembleAgent

agent = EnsembleAgent(collection)
agent.estimate(test[0].summary)